##Phase 1 : Loading the datasets


In [1]:
import json
import re
import os
import openpyxl
import networkx as nx

from collections import Counter, defaultdict
from functools import lru_cache

In [2]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_FOLDER = "/content/drive/My Drive/GILF"

GNP_PATH = f"{DRIVE_FOLDER}/GNP_FFF.xlsx"
CSP_PATH = f"{DRIVE_FOLDER}/CSP-FFF.xlsx"
OUTPUT_DIR = "/content/hindi_gender_kg/output"

Mounted at /content/drive


In [3]:
import os
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"GNP path exists: {os.path.exists(GNP_PATH)}")
print(f"CSP path exists: {os.path.exists(CSP_PATH)}")

GNP path exists: True
CSP path exists: True


In [4]:

import unicodedata

def normalise_hindi_text(value):
    """Normalise Unicode and whitespace without changing the wording."""
    if value is None:
        return ""
    text = unicodedata.normalize("NFC", str(value))
    return re.sub(r"\s+", " ", text).strip()


def normalise_header(value):
    return normalise_hindi_text(value).lower().replace("_", " ")


def resolve_columns(ws, required_aliases):
    """Resolve spreadsheet columns from flexible header aliases."""
    header_to_col = {
        normalise_header(ws.cell(1, col).value): col
        for col in range(1, ws.max_column + 1)
        if ws.cell(1, col).value is not None
    }

    resolved = {}
    for canonical, aliases in required_aliases.items():
        matches = [header_to_col[a] for a in aliases if a in header_to_col]
        if not matches:
            raise ValueError(
                f"Could not locate required column '{canonical}'. "
                f"Available headers: {list(header_to_col)}"
            )
        resolved[canonical] = matches[0]
    return resolved


def load_gnp(path):
    """Load, normalise and exactly deduplicate Gender-Neutral Pairs."""
    wb = openpyxl.load_workbook(path, data_only=True)
    ws = wb.active

    columns = resolve_columns(ws, {
        "original": {
            "original terms", "original term", "original", "gendered term"
        },
        "neutral": {
            "neutral terms", "neutral term", "neutral", "inclusive term"
        },
    })

    pairs = []
    seen = set()
    duplicate_rows = []

    for row_number in range(2, ws.max_row + 1):
        original = normalise_hindi_text(
            ws.cell(row_number, columns["original"]).value
        )
        neutral = normalise_hindi_text(
            ws.cell(row_number, columns["neutral"]).value
        )

        if not original or not neutral:
            continue

        key = (original, neutral)
        if key in seen:
            duplicate_rows.append({
                "row": row_number,
                "original": original,
                "neutral": neutral,
            })
            continue

        seen.add(key)
        pairs.append({
            "original": original,
            "neutral": neutral,
            "row_id": row_number - 1,
        })

    wb.close()
    return pairs, duplicate_rows


def load_csp(path):
    """Load, normalise and exactly deduplicate Counterfactual Sentence Pairs."""
    wb = openpyxl.load_workbook(path, data_only=True)
    ws = wb.active

    columns = resolve_columns(ws, {
        "pair_id": {"pair id", "pairid", "id"},
        "biased": {"biased sentence", "biased_sentence", "biased"},
        "counterfactual": {
            "counterfactual sentence", "counterfactual_sentence",
            "counterfactual", "inclusive sentence"
        },
    })

    pairs = []
    seen = set()
    duplicate_rows = []

    for row_number in range(2, ws.max_row + 1):
        pair_id = ws.cell(row_number, columns["pair_id"]).value
        biased = normalise_hindi_text(
            ws.cell(row_number, columns["biased"]).value
        )
        counterfactual = normalise_hindi_text(
            ws.cell(row_number, columns["counterfactual"]).value
        )

        if not biased or not counterfactual:
            continue

        key = (biased, counterfactual)
        if key in seen:
            duplicate_rows.append({
                "row": row_number,
                "biased": biased,
                "counterfactual": counterfactual,
            })
            continue

        seen.add(key)
        pairs.append({
            "pair_id": pair_id if pair_id is not None else row_number - 1,
            "biased": biased,
            "counterfactual": counterfactual,
        })

    wb.close()
    return pairs, duplicate_rows

In [5]:
OCC_KEYWORDS = [
    "कर्मी", "अधिकारी", "कारीगर", "प्रतिनिधि", "सेवक", "वितरक",
    "चालक", "कर्मचारी", "व्यापारी", "सहायक", "वाहक", "नाविक",
    "सैनिक", "प्रबंधक", "संपादक", "निदेशक", "शिक्षक", "कलाकार",
    "खिलाड़ी", "डॉक्टर", "पुलिस", "इंजीनियर", "वैज्ञानिक", "कवि",
    "लेखक", "पत्रकार", "वकील", "न्यायाधीश", "रसोइया", "चौकीदार",
    "माली", "धोबी", "नाई", "दर्जी", "किसान", "मज़दूर", "मजदूर",
]

REL_KEYWORDS = [
    "पति", "पत्नी", "भाई", "बहन", "माता", "पिता", "पुत्र", "पुत्री",
    "जीवनसाथी", "संतान", "अभिभावक", "विधवा", "विधुर", "दुल्हन",
    "दूल्हा", "कुँवारी", "कुँवारा", "अविवाहित", "माँ",
]

REP_KEYWORDS = [
    "मर्द", "औरत", "लड़का", "लड़की", "स्त्री", "पुरुष", "नारी",
    "महिला", "पागल", "दासी", "वेश्या", "रखैल", "छैला", "विलासिनी",
]


def classify_gnp_category(original, neutral):
    combined = f"{original} {neutral}"
    if any(keyword in combined for keyword in OCC_KEYWORDS):
        return "OCC"
    if any(keyword in combined for keyword in REL_KEYWORDS):
        return "REL"
    if any(keyword in combined for keyword in REP_KEYWORDS):
        return "REP"
    return "UNKNOWN"


def detect_grammatical_gender(term):
    """Conservative suffix-based lexical marker; context is handled later."""
    feminine_endings = (
        "वाली", "इनी", "िनी", "िका", "आनी", "ानी", "ित्री", "ंगना"
    )
    masculine_endings = ("वाला",)

    if term.endswith(feminine_endings):
        return "feminine_marked"
    if term.endswith(masculine_endings):
        return "masculine_marked"
    return "context_dependent"

In [6]:
GENDERED_SUBJECTS = {
    "masculine": [
        "पुरुषों", "पुरुष", "आदमियों", "आदमी", "लड़कों", "लड़के",
        "लड़का", "पिताओं", "पिता", "दूल्हों", "दूल्हा",
    ],
    "feminine": [
        "महिलाओं", "महिलाएँ", "महिला", "औरतों", "औरतें", "औरत",
        "लड़कियों", "लड़कियाँ", "लड़की", "माताओं", "माता", "माँ",
        "दुल्हनों", "दुल्हन",
    ],
}

ALL_SUBJECTS = [
    (subject, gender)
    for gender, subjects in GENDERED_SUBJECTS.items()
    for subject in subjects
]

DEVANAGARI_RANGE = r"\u0900-\u097F"


def contains_complete_hindi_phrase(text, phrase):
    """Match a complete Hindi token/phrase rather than an internal substring."""
    pattern = (
        rf"(?<![{DEVANAGARI_RANGE}])"
        rf"{re.escape(phrase)}"
        rf"(?![{DEVANAGARI_RANGE}])"
    )
    return re.search(pattern, normalise_hindi_text(text)) is not None


def extract_bias_predicate(biased_sentence):
    """Find a gendered group anywhere and retain a meaningful predicate."""
    sentence = normalise_hindi_text(biased_sentence)

    for subject, gender in sorted(ALL_SUBJECTS, key=lambda item: len(item[0]), reverse=True):
        pattern = (
            rf"(?<![{DEVANAGARI_RANGE}])"
            rf"{re.escape(subject)}"
            rf"(?![{DEVANAGARI_RANGE}])"
        )
        match = re.search(pattern, sentence)
        if not match:
            continue


        predicate = sentence[match.end():].strip() if match.start() == 0 else sentence
        return subject, gender, predicate

    return None, None, sentence


def detect_verb_gender(predicate):
    """Limited fallback markers used only as KG metadata, not as a parser."""
    feminine_markers = [
        "होती हैं", "होती है", "करती हैं", "करती है", "सकती हैं",
        "सकतीं", "रखती हैं", "नहीं होतीं", "नहीं होती", "चला सकतीं",
        "रख सकतीं", "नहीं सकतीं", "कर सकतीं", "रहनी चाहिए",
        "बनानी चाहिए", "लेनी चाहिए", "संभालनी चाहिए",
    ]
    masculine_markers = [
        "होते हैं", "होता है", "करते हैं", "करता है", "सकते हैं",
        "रखते हैं", "नहीं होते", "जा सकता", "चला सकते", "रख सकते",
        "नहीं सकते", "कर सकते", "रहने चाहिए", "बनाने चाहिए",
        "लेने चाहिए", "संभालने चाहिए",
    ]

    for marker in sorted(feminine_markers, key=len, reverse=True):
        if marker in predicate:
            return "feminine", marker
    for marker in sorted(masculine_markers, key=len, reverse=True):
        if marker in predicate:
            return "masculine", marker
    return "unknown", None


DIMENSION_KEYWORDS = {
    "Profession": ["नौकरी", "पेशा", "करियर", "व्यवसाय", "ऑफिस", "कंपनी"],
    "Family_Roles": [
        "परिवार", "शादी", "विवाह", "बच्च", "अभिभावक", "माता", "पिता",
        "माँ", "घर", "रसोई", "देखभाल", "ज़िम्मेदारी", "जिम्मेदारी",
    ],
    "Leadership_Traits": [
        "नेता", "नेतृत्व", "अध्यक्ष", "प्रबंधक", "बॉस", "निर्णय"
    ],
    "Emotion": [
        "भावना", "भावुक", "गुस्स", "रोना", "रो", "प्रेम", "दया",
        "सहानुभूति", "तर्कहीन", "कमज़ोरी", "कमजोरी",
    ],
    "Intelligence_Learning": [
        "बुद्धि", "गणित", "विज्ञान", "शोध", "पढ़", "अनपढ़", "शिक्ष",
        "ज्ञान", "सक्षम", "योग्य", "महारत",
    ],
    "Behavior": [
        "खेल", "ड्राइव", "गाड़ी", "साहस", "शक्ति", "मज़बूत",
        "मजबूत", "रंग", "गुलाबी", "रुचि",
    ],
}

DIMENSION_COUNTERCLAIMS = {
    "Profession": "व्यवसाय और कार्य-क्षमता लिंग पर निर्भर नहीं करती।",
    "Family_Roles": "परिवार और देखभाल की जिम्मेदारियाँ साझा की जा सकती हैं।",
    "Leadership_Traits": "नेतृत्व क्षमता और निर्णय-क्षमता लिंग पर निर्भर नहीं करती।",
    "Emotion": "भावनाएँ व्यक्त करना किसी भी व्यक्ति के लिए स्वाभाविक है।",
    "Intelligence_Learning": "बुद्धि, सीखने और शोध की क्षमता लिंग पर निर्भर नहीं करती।",
    "Behavior": "रुचि, व्यवहार और क्षमता किसी एक लिंग तक सीमित नहीं हैं।",
    "General": "किसी व्यक्ति की क्षमता या भूमिका लिंग से निर्धारित नहीं होती।",
}


def classify_stereotype_dimensions(text):
    text = normalise_hindi_text(text).lower()
    dimensions = [
        dimension
        for dimension, keywords in DIMENSION_KEYWORDS.items()
        if any(keyword in text for keyword in keywords)
    ]
    return dimensions or ["General"]


BIAS_CUE_PHRASES = [
    "सिर्फ़", "सिर्फ", "केवल", "का काम", "की ज़िम्मेदारी",
    "की जिम्मेदारी", "तक सीमित", "नहीं चाहिए", "नहीं बन",
    "नहीं कर", "नहीं होते", "नहीं होती", "नहीं सकती", "नहीं सकते",
    "कम सक्षम", "कमज़ोर", "कमजोर", "से कम",
]


def has_explicit_bias_cue(text):
    text = normalise_hindi_text(text)
    return any(
        contains_complete_hindi_phrase(text, cue)
        if " " not in cue
        else cue in text
        for cue in BIAS_CUE_PHRASES
    )
